# EDA — BIOSSES
Format: JSONL  
Task: Biomedical Semantic Textual Similarity  
Metric: Pearson r between cosine similarity and human scores  
Label: float 0.0 – 5.0 (human annotated)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')

DATA_DIR = Path('../data/raw/biosses')

def load_jsonl(filepath):
    rows = []
    with open(filepath) as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_jsonl(DATA_DIR / 'train.jsonl')
dev_df   = load_jsonl(DATA_DIR / 'validation.jsonl')
test_df  = load_jsonl(DATA_DIR / 'test.jsonl')

# cast label to float
for df in [train_df, dev_df, test_df]:
    df['label'] = df['label'].astype(float)

print(f'Train : {len(train_df)} pairs')
print(f'Dev   : {len(dev_df)} pairs')
print(f'Test  : {len(test_df)} pairs')
print(f'Total : {len(train_df)+len(dev_df)+len(test_df)} pairs')

## 2. Quick look

In [ ]:
train_df.head(10)

## 3. Label distribution

In [ ]:
all_df = pd.concat([train_df, dev_df, test_df], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(all_df['label'], bins=20, ax=axes[0], kde=True)
axes[0].set_title('Label distribution (all splits)')
axes[0].set_xlabel('similarity score (0-5)')

# per split boxplot
train_df['split'] = 'train'
dev_df['split']   = 'dev'
test_df['split']  = 'test'
combined = pd.concat([train_df, dev_df, test_df])
sns.boxplot(data=combined, x='split', y='label', ax=axes[1])
axes[1].set_title('Label distribution per split')

plt.tight_layout()
plt.show()

print(all_df['label'].describe().round(3))

## 4. Sentence length distribution

In [ ]:
all_df['len_1'] = all_df['text_1'].str.split().str.len()
all_df['len_2'] = all_df['text_2'].str.split().str.len()
all_df['len_diff'] = (all_df['len_1'] - all_df['len_2']).abs()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sns.histplot(all_df['len_1'], bins=20, ax=axes[0], kde=True, label='text_1')
sns.histplot(all_df['len_2'], bins=20, ax=axes[0], kde=True, label='text_2', alpha=0.6)
axes[0].set_title('Sentence length (words)')
axes[0].legend()

sns.histplot(all_df['len_diff'], bins=20, ax=axes[1], kde=True)
axes[1].set_title('Length difference between pairs')

axes[2].scatter(all_df['len_1'], all_df['label'], alpha=0.4, s=15)
axes[2].set_xlabel('sentence 1 length (words)')
axes[2].set_ylabel('similarity label')
axes[2].set_title('Length vs similarity')

plt.tight_layout()
plt.show()

print(all_df[['len_1','len_2','len_diff']].describe().round(2))

## 5. Label buckets — how many pairs are very similar vs very different?

In [ ]:
bins   = [0, 1, 2, 3, 4, 5]
labels = ['0-1 (unrelated)', '1-2 (low)', '2-3 (moderate)', '3-4 (high)', '4-5 (very similar)']
all_df['bucket'] = pd.cut(all_df['label'], bins=bins, labels=labels, include_lowest=True)

fig, ax = plt.subplots(figsize=(9, 4))
all_df['bucket'].value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_title('Pair count by similarity bucket')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

print(all_df['bucket'].value_counts().sort_index())

## 6. Sample pairs by score bucket

In [ ]:
for bucket in labels:
    sample = all_df[all_df['bucket']==bucket].sample(1, random_state=42).iloc[0]
    print(f'--- {bucket} (score={sample.label:.1f}) ---')
    print(f'  S1: {sample.text_1[:120]}')
    print(f'  S2: {sample.text_2[:120]}')
    print()

## 7. How we evaluate — Pearson r between cosine sim and human scores
This is what your benchmark script will compute. Here we show the expected range.

In [ ]:
from scipy.stats import pearsonr, spearmanr

# random baseline — what Pearson r would a random embedder get?
np.random.seed(42)
random_cosine = np.random.uniform(-1, 1, len(all_df))
r_random, _ = pearsonr(random_cosine, all_df['label'])

# perfect baseline — cosine sim perfectly matches labels
normalized = (all_df['label'] - all_df['label'].min()) / (all_df['label'].max() - all_df['label'].min()) * 2 - 1
r_perfect, _ = pearsonr(normalized, all_df['label'])

print(f'Random embedder Pearson r : {r_random:.4f}  (floor — anything below this is broken)')
print(f'Perfect embedder Pearson r: {r_perfect:.4f}  (ceiling)')
print()
print('Published baselines for reference:')
print('  Word2Vec (Google News) : ~0.60')
print('  BioWordVec             : ~0.85')
print('  SapBERT                : ~0.90')

## 8. Summary

In [ ]:
pd.DataFrame({
    'split'     : ['train', 'dev', 'test', 'total'],
    'pairs'     : [len(train_df), len(dev_df), len(test_df), len(all_df)],
    'mean_score': [train_df['label'].mean(), dev_df['label'].mean(),
                   test_df['label'].mean(),  all_df['label'].mean()],
    'std_score' : [train_df['label'].std(), dev_df['label'].std(),
                   test_df['label'].std(),  all_df['label'].std()],
}).round(3).set_index('split')